# Setup

In [47]:
import os
from pathlib import Path

if (Path(os.getcwd()) != Path("c:/Users/alexi/studcess/api")):
    os.chdir("c:/Users/alexi/studcess/api")

from dotenv import load_dotenv

EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
GENERATIVE_MODEL = ""
K = ""

# Define classes
Class definitions for objects that will be reused often:
- Document: object for the parent document. 
- Chunks: object for chunks

In [48]:
from enum import Enum

class ContentType(str,Enum):
    THEORY = "theory"
    EXERCISES = "exercise"
    EXAM = "exam"
    
class Course(str,Enum):
    CALCULUS = "calculus",
    PROGRAMMING = "programming"

In [ ]:
from docling_core.types.doc.document import DoclingDocument
from dataclasses import dataclass
from dataclasses import field

@dataclass
class Document:
    path: Path
    title: str
    course: str
    language: str
    content_type: ContentType
    docling_doc: DoclingDocument | None = None
        
@dataclass
class Chunk:
    document: Document
    text: str | None = None
    pages: set[int] | None = None
    embedding: list[float] | None = None
    heading_path: str | None = None # Path of md headings: Chapter 4 > Lists > List comprehensions

    def to_row(self, doc: Document) -> dict:
        return {
            "content": self.text,
            "embedding": self.embedding,
            "content_type": doc.content_type,   # inherited from the document
        }

# Load the document

## Working assumption
We assume that documents loaded to Studcess are **structured**, i.e. we know exactly what the content type, and course are as the user will be forced to specify in the UI so that we do not have to guess.

In [50]:
docs = Path("sample-data")
sample_path = docs / "cours_ana1.pdf" 

sample = Document(
    path=sample_path,
    title=sample_path.stem,
    course=Course.CALCULUS,
    content_type=ContentType.THEORY,
    language="fr"
)

print(sample)

Document(path=WindowsPath('sample-data/cours_ana1.pdf'), title='cours_ana1', course=<Course.CALCULUS: 'calculus'>, language='fr', content_type=<ContentType.THEORY: 'theory'>, docling_doc=None, chunks=[])


# Run docling on the document

In [51]:
from docling.datamodel.service import ConvertDocumentsOptions
from langchain_docling.loader import ExportType
from langchain_docling import DoclingLoader
from docling.datamodel.pipeline_options import PdfPipelineOptions, HeadingHierarchyOptions, OcrOptions, OcrAutoOptions
from docling.datamodel.base_models import InputFormat
from docling.document_converter import PdfFormatOption, DocumentConverter

pdf_pipeline_options = PdfPipelineOptions(
    do_ocr=True,
    do_table_structure=True,
    generate_page_images=False,
    do_picture_description=False,
    do_formula_enrichment=True,
    heading_hierarchy_options=HeadingHierarchyOptions(enabled=True),
    ocr_options=OcrAutoOptions(
        lang=[sample.language],
        replace_broken_text_layer=True
    )
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_pipeline_options),
    },
    show_progress=True
)

result = converter.convert(sample.path)
docling_doc = result.document
sample.docling_doc = docling_doc

[1] Converting cours_ana1.pdf
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 2507.15it/s]
c:\Users\alexi\studcess\api\.venv\Lib\site-packages\transformers\configuration_utils.py:726: FutureWarning: `rope_scaling` is deprecated and will be removed in a future version. Set `rope_parameters` instead, with `rope_type` and `rope_theta` included.
  warnings.warn(
Loading weights: 100%|██████████| 471/471 [00:00<00:00, 1301.60it/s]
[transformers] The tied weights mapping and config for this model specifies to tie model.text_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
  pages 14/14
  code and formulas 30/30
Finished cours_ana1.pdf: success


# Cleanup

## Fix encoding errors

In [69]:
from docling_core.types.doc import DocItemLabel
from ftfy import fix_encoding

for item in docling_doc.texts:
    if item.label not in [DocItemLabel.FORMULA,DocItemLabel.CODE]:
        item.text = fix_encoding(item.text)

# Splitting the document into chunks

For each chunk:

In [70]:
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer

tokenizer = HuggingFaceTokenizer.from_pretrained(
    model_name=EMBEDDING_MODEL,
    max_tokens=512
)

chunker = HybridChunker(
    tokenizer=tokenizer,
    merge_peers=True
)

for i,chunk in enumerate(chunker.chunk(docling_doc)):
    headings = chunk.meta.headings
    
    pages = []
    
    for item in chunk.meta.doc_items:
        for p in item.prov:
            pages.append(p.page_no)
    
    pages = set(pages)
    
    print(f"CHUNK {i}")
    print("Headings",headings)
    print("Pages",pages)
    print(chunk.text)

CHUNK 0
Headings ['Analyse 1', 'Table des matières']
Pages {1}
| - |
| , 1 L'addition et la multiplication = nombres. , 1 L'addition et la multiplication = nombres. , 3 = . 2, 1 L'addition et la multiplication = Les ensembles de. 2, 1 L'addition et la multiplication = . 2, 3 = 4. , 1 L'addition et la multiplication = 2.1. , 1 L'addition et la multiplication = Ensembles de nombres . . . . . . . . . . . . . . . . .. , 3 = . . . . . . . . . . . . . . . . . . 4. , 1 L'addition et la multiplication = 2.2. , 1 L'addition et la multiplication = Op´erations sur les ensembles . . . . . . . . . . . . .. , 3 = . . . . . . . . . . . . . . . . . . 4. , 1 L'addition et la multiplication = 2.3. , 1 L'addition et la multiplication = Intervalles . . . . . . . . . . . . . . . . . . . . . . . .. , 3 = . . . . . . . . . . . . . . . . . . 4. , 1 L'addition et la multiplication = 2.4. , 1 L'addition et la multiplication = Valeur absolue . . . . . . . . . . . . . . . . . . . . .. , 3 = . . . . . . . . . . . 